In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle

# --- Load Data and Create a Unified DataFrame ---
PKL_PATH = '../../data/main_figures/slr_res/DIVA_data/Coast_with_Mang.pkl' 
COASTLINE_DATA_PATH = "../../data/main_figures/slr_res/DIVA_coastline_WGS84_Point.csv"
with open(PKL_PATH, 'rb') as f:
    data = pickle.load(f)

# # Extract relevant data
df = pd.DataFrame({
    'ClsFID': data['ClsFID'],
    'area': data['area'],
    'agb_present': data['agb_present'],
    'agb_grow': data['agb_grow'],
    'agb_126': data['agb_126'],
    'agb_585': data['agb_585'],
    # 注意：根据您的脚本，这里选择的是第2列(Pop5)和第1列(50%恢复)
    'area_after_SLR_126': data['area_after_SLR_126'][:, 1], 
    'area_after_SLR_585': data['area_after_SLR_585'][:, 1],
    'area_restoration_126': data['area_restoration_126'][:, 0],
    'area_restoration_585': data['area_restoration_585'][:, 0],
})

# --- Load and Merge Country Information ---
# 从新的基础数据文件中读取国家信息
coastline_data = pd.read_csv(COASTLINE_DATA_PATH)
# 使用.merge()方法，通过'ClsFID'将国家信息精确地匹配到每一行
# *** FIX: Parse the 'ID_Country' column to extract the clean country name ***
# Split the column on the first space. expand=True creates new columns.
country_split = coastline_data["ID_Country"].str.split(" ", n=1, expand=True)
# Create the country_info DataFrame for merging.
# We take the original CLSFID for merging and the newly extracted country name (column 1).
country_info = pd.DataFrame({'ClsFID': coastline_data['CLSFID'], 'Country': country_split[1]})
# 将主数据框与国家信息合并
df = pd.merge(df, country_info, on='ClsFID', how='left')

# --- Calculate AGB Changes (Vectorized Operations) ---
# 定义一个函数，使其在DataFrame上操作，更清晰
def calculate_agb_changes(df, scenario_suffix):
    agb_scen = f'agb_{scenario_suffix}'
    area_slr_scen = f'area_after_SLR_{scenario_suffix}'
    area_res_scen = f'area_restoration_{scenario_suffix}'
    # 直接在DataFrame上创建新列，单位转换为 Tg (10^6 Mg)
    df[f'total_change_{scenario_suffix}'] = (df[agb_scen] * (df[area_slr_scen] + df[area_res_scen]) - df['agb_present'] * df['area']) / 1e6
    df[f'grow_change_{scenario_suffix}'] = (df['agb_grow'] * df['area'] - df['agb_present'] * df['area']) / 1e6
    df[f'ht_change_{scenario_suffix}'] = (df[agb_scen] * df['area'] - df['agb_grow'] * df['area']) / 1e6
    df[f'res_change_{scenario_suffix}'] = (df[agb_scen] * df[area_res_scen]) / 1e6
    df[f'slr_change_{scenario_suffix}'] = (df[agb_scen] * (df[area_slr_scen] - df['area'])) / 1e6
    return df
df = calculate_agb_changes(df, '126')
df = calculate_agb_changes(df, '585')

# --- Aggregate Results by Country using groupby ---
agg_cols_126 = {
    'AGB_Change': 'total_change_126',
    'AGB_Change_Grow': 'grow_change_126',
    'AGB_Change_HT': 'ht_change_126',
    'AGB_Change_RES': 'res_change_126',
    'AGB_Change_SLR': 'slr_change_126'
}
agg_cols_585 = {
    'AGB_Change': 'total_change_585',
    'AGB_Change_Grow': 'grow_change_585',
    'AGB_Change_HT': 'ht_change_585',
    'AGB_Change_RES': 'res_change_585',
    'AGB_Change_SLR': 'slr_change_585'
}
# *** FIX: Invert the dictionaries for the rename function ***
rename_map_126 = {v: k for k, v in agg_cols_126.items()}
rename_map_585 = {v: k for k, v in agg_cols_585.items()}

# 使用groupby().sum()一步完成按国家汇总，然后使用翻转后的字典进行重命名
country_agb_change_126 = df.groupby('Country', as_index=False)[list(agg_cols_126.values())].sum().rename(columns=rename_map_126).sort_values(by='AGB_Change')
country_agb_change_585 = df.groupby('Country', as_index=False)[list(agg_cols_585.values())].sum().rename(columns=rename_map_585).sort_values(by='AGB_Change')

# country_agb_change_126.to_csv('../../data/main_figures/slr_res/output/country_agb_change_126.csv', index=False)
# country_agb_change_585.to_csv('../../data/main_figures/slr_res/output/country_agb_change_585.csv', index=False)

In [ ]:
# Plot AGB changes by country
data1 = country_agb_change_126.iloc[:5]
data2 = country_agb_change_126.iloc[-5:].iloc[::-1]
data3 = country_agb_change_585.iloc[:5]
data4 = country_agb_change_585.iloc[-5:].iloc[::-1]

# Set global font to Helvetica
plt.rcParams['font.family'] = 'Helvetica'
plt.rcParams['font.size'] = 16
# Set the mathtext font to Helvetica
plt.rcParams['mathtext.fontset'] = 'custom'
plt.rcParams['mathtext.rm'] = 'Helvetica'
plt.rcParams['mathtext.it'] = 'Helvetica:italic'
plt.rcParams['mathtext.bf'] = 'Helvetica:bold'


fig, axs = plt.subplots(2, 2, figsize=(12, 9))
fig.text(0.01, 0.5, 'AGB Change (Tg DM)', va='center', rotation='vertical', fontsize=18)

ax1 = axs[0, 0]
AGB_change = data1[['AGB_Change_Grow', 'AGB_Change_HT', 'AGB_Change_RES', 'AGB_Change_SLR']]
AGB_change.plot(kind='bar', stacked=True, ax=ax1, color=['#76d7c4', '#f3b6b1', '#ebdef7', '#aad6f1'])
ax1.scatter(range(0, 5), data1.iloc[:, 1], facecolor='none', edgecolor='black', s=100, label='Total')

ax1.set_xticklabels(data1['Country'], rotation=35, ha='right')
ax1.set_ylim([-75, 25])
ax1.legend().set_visible(False)
# handles, _ = ax1.get_legend_handles_labels()
# labels = ['Total', 'Potential growth to mature', 'Hydrotherm', 'Restoration', 'Sea-level rise']
# ax1.legend(handles, labels, loc='upper right', frameon=False, labelspacing=0.12, bbox_to_anchor=(1, 0.6), handlelength=0.8, handletextpad=0.3)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
ax1.spines['bottom'].set_visible(False)
ax1.set_title(f"Largest AGB decrease countries\n$\mathbf{{{chr(97)}}}$", loc='left', fontsize=18)

ax2 = axs[0, 1]
AGB_change = data2[['AGB_Change_Grow', 'AGB_Change_HT', 'AGB_Change_RES', 'AGB_Change_SLR']]
AGB_change.plot(kind='bar', stacked=True, ax=ax2, color=['#76d7c4', '#f3b6b1', '#ebdef7', '#aad6f1'])
ax2.scatter(range(0, 5), data2.iloc[:, 1], facecolor='none', edgecolor='black', s=100, label='Total')

ax2.set_xticklabels(data2['Country'], rotation=35, ha='right')
ax2.set_ylim([-50, 100])
ax2.legend().set_visible(False)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
ax2.spines['bottom'].set_visible(False)
ax2.set_title(f"Largest AGB increase countries\n$\mathbf{{{chr(98)}}}$", loc='left', fontsize=18)

ax2_right = ax2.twinx()
ax2_right.set_ylabel('Low-warming', rotation=270, color='k', labelpad=20, fontsize=18)
ax2_right.set_yticks([])
ax2_right.spines['right'].set_visible(False)
ax2_right.spines['top'].set_visible(False)
ax2_right.spines['left'].set_visible(False)
ax2_right.spines['bottom'].set_visible(False)

ax3 = axs[1, 0]
AGB_change = data3[['AGB_Change_Grow', 'AGB_Change_HT', 'AGB_Change_RES', 'AGB_Change_SLR']]
AGB_change.plot(kind='bar', stacked=True, ax=ax3, color=['#76d7c4', '#f3b6b1', '#ebdef7', '#aad6f1'])
ax3.scatter(range(0, 5), data3.iloc[:, 1], facecolor='none', edgecolor='black', s=100, label='Total')

ax3.set_xticklabels(data3['Country'], rotation=35, ha='right')
ax3.set_ylim([-340, 160])
ax3.legend().set_visible(False)
ax3.spines['top'].set_visible(False)
ax3.spines['right'].set_visible(False)
ax3.spines['bottom'].set_visible(False)
ax3.set_title(f"$\mathbf{{{chr(99)}}}$", loc='left', fontsize=18)

ax4 = axs[1, 1]
AGB_change = data4[['AGB_Change_Grow', 'AGB_Change_HT', 'AGB_Change_RES', 'AGB_Change_SLR']]
AGB_change.plot(kind='bar', stacked=True, ax=ax4, color=['#76d7c4', '#f3b6b1', '#ebdef7', '#aad6f1'])
ax4.scatter(range(0, 5), data4.iloc[:, 1], facecolor='none', edgecolor='black', s=100, label='Total')

ax4.set_xticklabels(data4['Country'], rotation=35, ha='right')
ax4.set_ylim([-20, 80])
# ax4.legend().set_visible(False)
handles, _ = ax1.get_legend_handles_labels()
labels = ['Total', 'Potential growth to maturity', 'Climate and hydrology', 'Restoration', 'Sea-level rise']
ax4.legend(handles, labels, loc='upper right', frameon=False, labelspacing=0.12, bbox_to_anchor=(1, 1.1), handlelength=0.8, handletextpad=0.3)
ax4.spines['top'].set_visible(False)
ax4.spines['right'].set_visible(False)
ax4.spines['bottom'].set_visible(False)
ax4.set_title(f"$\mathbf{{{chr(100)}}}$", loc='left', fontsize=18)

ax4_right = ax4.twinx()
ax4_right.set_ylabel('High-warming', rotation=270, color='k', labelpad=20, fontsize=18)
ax4_right.set_yticks([])
ax4_right.spines['right'].set_visible(False)
ax4_right.spines['top'].set_visible(False)
ax4_right.spines['left'].set_visible(False)
ax4_right.spines['bottom'].set_visible(False)

plt.tight_layout()
plt.savefig('py04_country_change.png', dpi=300, transparent=True)

In [ ]:
# --- 5. Print Final Results ---
# Set the display format for floating-point numbers to 2 decimal places
pd.set_option('display.float_format', '{:.2f}'.format)
print("SSP1-2.6")
print("Largest decrease:")
# FIX: Select only 'Country' and 'AGB_Change' columns for printing
print(country_agb_change_126[['Country', 'AGB_Change']].head().to_string(index=False))
print("Largest increase:")
# FIX: Select only 'Country' and 'AGB_Change' columns for printing
print(country_agb_change_126[['Country', 'AGB_Change']].tail().to_string(index=False))

In [ ]:
print("SSP5-8.5")
print("Largest decrease:")
# FIX: Select only 'Country' and 'AGB_Change' columns for printing
print(country_agb_change_585[['Country', 'AGB_Change']].head().to_string(index=False))
print("Largest increase:")
# FIX: Select only 'Country' and 'AGB_Change' columns for printing
print(country_agb_change_585[['Country', 'AGB_Change']].tail().to_string(index=False))

In [ ]:
# The unit is already Mg, we sum it up and convert to Tg
df['agb_present_total'] = df['agb_present'] * df['area'] / 1e6
country_present_agb = df.groupby('Country', as_index=False)['agb_present_total'].sum()

# Merge present-day AGB with the change data
country_analysis_126 = pd.merge(country_agb_change_126, country_present_agb, on='Country')
country_analysis_585 = pd.merge(country_agb_change_585, country_present_agb, on='Country')

import matplotlib.pyplot as plt
from brokenaxes import brokenaxes
from adjustText import adjust_text

# --- Set plotting style ---
plt.rcParams['font.family'] = 'Helvetica'
plt.rcParams['font.size'] = 16

# --- Create the figure ---
fig = plt.figure(figsize=(12, 6))
fig.supxlabel('Present-day Mangrove AGB (Tg DM)', y=0, fontsize=16)

# --- Define constants ---
break_point = 190
xlims = ((0, break_point), (490, 540))
top_10_countries = country_analysis_126.sort_values(by='agb_present_total', ascending=False).head(10)['Country'].tolist()

# --- Create the broken axes objects ---
gs = fig.add_gridspec(1, 2, wspace=0.1)
bax1 = brokenaxes(xlims=xlims, subplot_spec=gs[0, 0])
bax2 = brokenaxes(xlims=xlims, subplot_spec=gs[0, 1])

# --- Plot data on the broken axes ---
bax1.scatter(country_analysis_126['agb_present_total'], country_analysis_126['AGB_Change'], alpha=0.8, color='royalblue', edgecolors='k', s=60, zorder=10)
bax1.axhline(0, color='grey', linestyle='--', linewidth=1.2)
bax1.set_ylabel('AGB Change (Tg DM)', labelpad=50) # Reduced labelpad for better look
bax1.set_title('a', loc='left', fontsize=16, fontweight='bold')
bax1.grid(linestyle=':', alpha=0.7)
bax1.text(0.05, 0.95, 'Low-warming scenario', transform=bax1.axs[0].transAxes, ha='left', fontsize=16)

bax2.scatter(country_analysis_585['agb_present_total'], country_analysis_585['AGB_Change'], alpha=0.8, color='firebrick', edgecolors='k', s=60, zorder=10)
bax2.axhline(0, color='grey', linestyle='--', linewidth=1.2)
bax2.set_title('b', loc='left', fontsize=16, fontweight='bold')
bax2.grid(linestyle=':', alpha=0.7)
bax2.text(0.05, 0.95, 'High-warming scenario', transform=bax2.axs[0].transAxes, ha='left', fontsize=16)
bax2.tick_params(axis='y', labelleft=False)

# --- Synchronize the Y-axis limits ---
y1_min, y1_max = bax1.axs[0].get_ylim()
y2_min, y2_max = bax2.axs[0].get_ylim()
bax1.set_ylim(bottom=min(y1_min, y2_min), top=max(y1_max, y2_max)+20)
bax2.set_ylim(bottom=min(y1_min, y2_min), top=max(y1_max, y2_max)+20)

# --- REFACTORED HYBRID LABELING STRATEGY ---

# *** VVVV 您唯一需要交互的地方 VVVV ***
manual_overrides = {
    'low_warming': {
        'Myanmar': {'xytext': (15, 15)},
        'Bangladesh': {'xytext': (-20, 10)},
        'Colombia': {'xytext': (0, -25)},
        'Mexico': {'xytext': (0, -25)},
        'Venezuela': {'xytext': (15, -25)},
        'Brazil': {'xytext': (0, -25)},
        'Papua New Guinea': {'xytext': (45, 7)},
    },
    'high_warming': {
        'Bangladesh': {'xytext': (15, 35)},
        'Colombia': {'xytext': (15, -35)},
        'Venezuela': {'xytext': (15, -25)},
        'Malaysia': {'xytext': (15, 5)},
        'Papua New Guinea': {'xytext': (75, 5)},
    }
}
# *** ^^^^ 您只需要在这里修改 ^^^^ ***

def process_labels(df, bax, overrides_key, break_point):
    """
    A reusable function to handle both manual and automatic labeling for a given plot.
    """
    texts_to_adjust = []
    points_to_avoid = df[df['agb_present_total'] < break_point]
    overrides = manual_overrides.get(overrides_key, {})

    for _, row in df.iterrows():
        if row['Country'] in top_10_countries:
            source_ax = bax.axs[1] if row['agb_present_total'] > break_point else bax.axs[0]
            if row['Country'] in overrides:
                # Manual placement
                override = overrides[row['Country']]
                source_ax.annotate(row['Country'], xy=(row['agb_present_total'], row['AGB_Change']),
                                   xytext=override.get('xytext', (5, 5)), textcoords='offset points',
                                   ha='center', va='bottom', fontsize=12,
                                   arrowprops=dict(arrowstyle="-", color='black', lw=0.5))
            elif row['agb_present_total'] < break_point:
                # Add to auto-adjust list (crowded area)
                texts_to_adjust.append(source_ax.text(row['agb_present_total'], row['AGB_Change'], row['Country'], fontsize=12))
            else:
                # Direct placement (uncrowded area)
                source_ax.text(row['agb_present_total'], row['AGB_Change'] + 2, row['Country'], ha='center', va='bottom', fontsize=12)
    
    return texts_to_adjust, points_to_avoid

# --- Process both plots using the new function ---
texts_126, points_126 = process_labels(country_analysis_126, bax1, 'low_warming', break_point)
texts_585, points_585 = process_labels(country_analysis_585, bax2, 'high_warming', break_point)

# --- Run adjust_text for each plot ---
adjust_text(texts_126, x=points_126['agb_present_total'], y=points_126['AGB_Change'],
            ax=bax1.axs[0], expand_points=(2.5, 2.5),
            arrowprops=dict(arrowstyle="-", color='black', lw=0.5))

adjust_text(texts_585, x=points_585['agb_present_total'], y=points_585['AGB_Change'],
            ax=bax2.axs[0], expand_points=(2.5, 2.5),
            arrowprops=dict(arrowstyle="-", color='black', lw=0.5))

plt.savefig('py04b_country_change_to_present_total.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# --- 1. 确定AGB总量前十的国家 ---
top_10_countries_by_total_agb = country_analysis_126.sort_values(
    by='agb_present_total', ascending=False
).head(10)['Country'].tolist()

# --- 2. 准备用于绘图的数据 ---
# 按当前AGB总量降序排列 (从大到小)
data_for_plot_126 = country_analysis_126[
    country_analysis_126['Country'].isin(top_10_countries_by_total_agb)
].sort_values(by='agb_present_total', ascending=False)

data_for_plot_585 = country_analysis_585[
    country_analysis_585['Country'].isin(top_10_countries_by_total_agb)
].sort_values(by='agb_present_total', ascending=False)

# --- 3. 创建图表和子图 (2行1列，共享X轴) ---
fig_new, axs_new = plt.subplots(2, 1, figsize=(12, 12), sharex=True)
fig_new.supylabel('AGB Change (Tg DM)', fontsize=16)

# --- 定义一个可复用的绘图函数 ---
def plot_top10_bars(ax, df_data, title_char, scenario_text, show_legend=False):
    """一个函数来绘制每个子图，避免代码重复"""
    
    change_components = df_data[['AGB_Change_Grow', 'AGB_Change_HT', 'AGB_Change_RES', 'AGB_Change_SLR']]
    bar_colors = ['#76d7c4', '#f3b6b1', '#ebdef7', '#aad6f1']

    change_components.plot(kind='bar', stacked=True, ax=ax, color=bar_colors, edgecolor='none', width=0.7, legend=False)
    ax.scatter(range(len(df_data)), df_data['AGB_Change'], facecolor='none', edgecolor='black', s=100, zorder=5)
    ax.axhline(0, color='grey', linestyle='--', linewidth=1, zorder=0)

    ax.set_title(title_char, loc='left', fontsize=16, fontweight='bold')
    ax.text(0.02, 0.95, scenario_text, transform=ax.transAxes, ha='left', va='top', fontsize=16)

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_visible(False)
    
    ax.tick_params(axis='y')
    ax.set_xlabel('')
    
    if show_legend:
        bar_labels = ['Potential growth to maturity', 'Climate and hydrology change', 'Restoration', 'Sea-level rise']
        bar_handles = [plt.Rectangle((0,0),1,1, color=color) for color in bar_colors]
        scatter_handle = plt.Line2D([0], [0], marker='o', color='k', linestyle='None', markersize=8, fillstyle='none', label='Total')
        
        all_handles = [scatter_handle] + bar_handles
        all_labels = ['Total'] + bar_labels
        
        ax.legend(handles=all_handles, labels=all_labels,
                  loc='lower right', frameon=False, labelspacing=0.6)

# --- 绘制上下两个子图 ---
plot_top10_bars(axs_new[0], data_for_plot_126, 'a', 'Low-warming scenario', show_legend=False)
plot_top10_bars(axs_new[1], data_for_plot_585, 'b', 'High-warming scenario', show_legend=True)

# *** 修正：独立调整每个子图的Y轴范围 ***

# 调整上方子图 (Low-warming)
y1_min, y1_max = axs_new[0].get_ylim()
data_range_1 = y1_max - y1_min
adjusted_ymax_1 = y1_max + data_range_1 * 0.1
axs_new[0].set_ylim(bottom=y1_min, top=adjusted_ymax_1)

# 调整下方子图 (High-warming)
y2_min, y2_max = axs_new[1].get_ylim()
data_range_2 = y2_max - y2_min
adjusted_ymax_2 = y2_max + data_range_2 * 0.1
axs_new[1].set_ylim(bottom=y2_min, top=adjusted_ymax_2)


# --- 5. 设置共享的X轴 ---
plt.sca(axs_new[1])
plt.xticks(ticks=range(len(data_for_plot_585)), labels=data_for_plot_585['Country'], rotation=45, ha='right')
plt.tick_params(axis='x', length=5)

# --- 6. 最终布局调整 ---
plt.tight_layout()

# --- 保存并显示图像 ---
plt.savefig('py04c_top10_by_totalAGB_categorized_change_final.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
from brokenaxes import brokenaxes
import numpy as np

import numpy as np
import matplotlib.pyplot as plt

np.random.seed(19680801)

pts = np.random.rand(30)*.2
pts[[3, 14]] += 1.8 #将索引为3个和14的元素加1.8处理成两个离散点


fig = plt.figure(dpi=120)
bax = brokenaxes(xlims=((0, 10), (11, 20)),
                 ylims=((0, 0.28), (0.4, 2)),#设置y轴裂口范围
                 hspace=0.25,#y轴裂口宽度
                 wspace=0.2,#x轴裂口宽度
                 despine=False,#是否y轴只显示一个裂口
                 diag_color='r',#裂口斜线颜色
                 )
bax.plot(pts)
plt.show()